# Sistema adaptativo de deteccion de fraude: IEEE-CIS Fraud Detection

Proyecto 1, Planificacion y Toma de Decisiones en IA (UTEC). Sistema
inteligente adaptativo con datos no estacionarios para deteccion de fraude.

Este notebook tiene dos partes. La primera (secciones 1 a 10) es el analisis
exploratorio: responde si el dataset esta listo para modelar, o si hay
riesgos de fuga de datos, variables ruidosas, datos sucios o datos faltantes
que primero hay que resolver. La segunda (secciones 11 en adelante) usa esas
conclusiones para construir el pipeline completo: ingenieria de variables,
comparacion de modelos, una politica de decision basada en costo, monitoreo
de degradacion temporal y una estrategia de reentrenamiento adaptativo,
siguiendo los cuatro objetivos especificos del proyecto. Cada seccion trae un
grafico, una lectura de ese grafico en texto corrido, y termina en
conclusiones concretas y accionables.

## Contenido

1. Objetivo y alcance
2. Carga y union de datos
3. Auditoria estructural
4. Datos faltantes
5. Consistencia y valores atipicos
6. Riesgo de fuga de datos
7. Redundancia y variables ruidosas
8. Balance de clases y variacion temporal
9. Equidad y limitaciones
10. Conclusiones del EDA
11. Del EDA al pipeline: decisiones y supuestos
12. Split temporal con gap de latencia de etiqueta
13. Ingenieria de variables
14. Manejo del desbalance de clases
15. Modelos, metricas y conclusiones (Objetivo 1)
16. Verificacion empirica de fuga por cliente
17. Politica de decision basada en costo (Objetivo 2)
18. Monitoreo de degradacion temporal (Objetivo 3)
19. Adaptacion: reentrenamiento con ventana deslizante (Objetivo 4)
20. Conclusiones finales

## 1. Objetivo y alcance

El dataset combina `train_transaction.csv` y `train_identity.csv`, unidos por
`TransactionID` mediante left join, porque no todas las transacciones tienen
fila de identidad asociada. La variable objetivo es `isFraud` (0 = legitima,
1 = fraude), con un desbalance conocido de aproximadamente 96.5% contra 3.5%.
El dataset tiene un componente temporal explicito a traves de `TransactionDT`,
y el proyecto exige un split cronologico, nunca aleatorio, para simular
llegada de datos en el tiempo y medir concept drift. La metrica principal es
PR-AUC, complementada con recall, precision y F1.

Las columnas `C`, `D`, `V` y `M` estan anonimizadas: Vesta no publica su
significado exacto. Cualquier lectura sobre ellas en este notebook es un
hallazgo estadistico, no una interpretacion causal de negocio.

## 2. Carga y union de datos

In [ ]:
import gc
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

DATA_DIR = Path("/kaggle/input/competitions/ieee-fraud-detection")
WORK_DIR = Path("/kaggle/working")
progress = {"status": "running", "stage": "start", "artifacts": []}


def save_progress(stage):
    progress["stage"] = stage
    (WORK_DIR / "run_summary.json").write_text(json.dumps(progress, indent=2), encoding="utf-8")


def save_artifact(obj, name):
    path = WORK_DIR / name
    joblib.dump(obj, path)
    progress["artifacts"].append(name)


def barh(series, title, xlabel, figsize=(9, 6), color="#4C72B0"):
    fig, ax = plt.subplots(figsize=figsize)
    series.sort_values().plot(kind="barh", ax=ax, color=color)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    plt.tight_layout()
    plt.show()


save_progress("setup")

Las columnas `C`, `D` y `V` son casi 360 de las 434 columnas finales, asi
que se leen como `float32` en vez de `float64` desde el CSV: reduce el uso de
memoria a la mitad sin perder precision util para este analisis.

In [ ]:
dtype_txn = {"TransactionID": "int32", "isFraud": "int8", "TransactionDT": "int32",
             "TransactionAmt": "float32"}
for i in range(1, 15):
    dtype_txn[f"C{i}"] = "float32"
for i in range(1, 16):
    dtype_txn[f"D{i}"] = "float32"
for i in range(1, 340):
    dtype_txn[f"V{i}"] = "float32"
for c in ["card1", "card2", "card3", "card5", "addr1", "addr2", "dist1", "dist2"]:
    dtype_txn[c] = "float32"

txn = pd.read_csv(DATA_DIR / "train_transaction.csv", dtype=dtype_txn)
for c in ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain"] + [f"M{i}" for i in range(1, 10)]:
    txn[c] = txn[c].astype("category")

idn = pd.read_csv(DATA_DIR / "train_identity.csv")
idn["TransactionID"] = idn["TransactionID"].astype("int32")
for c in idn.columns:
    if c == "TransactionID":
        continue
    idn[c] = idn[c].astype("float32") if pd.api.types.is_numeric_dtype(idn[c]) else idn[c].astype("category")

df = txn.merge(idn, on="TransactionID", how="left")
del txn, idn
gc.collect()

id_device_cols = [c for c in df.columns if c.startswith("id_") or c in ("DeviceType", "DeviceInfo")]
v_cols = [c for c in df.columns if c.startswith("V")]
c_cols = [c for c in df.columns if c.startswith("C") and c[1:].isdigit()]
d_cols = [c for c in df.columns if c.startswith("D") and c[1:].isdigit()]

print("Forma final tras el merge:", df.shape)
save_progress("datos_cargados")

## 3. Auditoria estructural

Antes de mirar contenido, se verifica la forma del dataset: cuantas filas y
columnas hay, si `TransactionID` funciona realmente como llave unica de
union, si hay filas duplicadas, y como se distribuyen las variables
categoricas.

In [ ]:
dup_exactos = int(pd.util.hash_pandas_object(df, index=False).duplicated().sum())
rango_dias = (df["TransactionDT"].max() - df["TransactionDT"].min()) / (3600 * 24)

print(f"Filas: {df.shape[0]:,}   Columnas: {df.shape[1]}")
print(f"TransactionID es unico: {df['TransactionID'].is_unique}")
print(f"Duplicados exactos (hash de fila completa): {dup_exactos}")
print(f"Rango temporal cubierto: {rango_dias:.1f} dias")

In [ ]:
cat_cols_p1 = ["ProductCD", "card4", "card6"] + [f"M{i}" for i in range(1, 10)] + \
              ["DeviceType", "P_emaildomain", "R_emaildomain"]
cardinalidad = pd.Series({c: df[c].nunique(dropna=True) for c in cat_cols_p1})
barh(cardinalidad, "Cardinalidad de variables categoricas clave", "cantidad de valores unicos")

No hay filas duplicadas y `TransactionID` es unico en las 590,540 filas:
se descarta como feature y se usa unicamente como llave de union. El rango
temporal de 182 dias coincide con lo documentado para el set de
entrenamiento, lo que confirma que `TransactionDT` debe tratarse como
variable temporal para ordenar y particionar, no como una feature numerica
que se entrega cruda al modelo. Entre las categoricas, `P_emaildomain` y
`R_emaildomain` concentran la mayor cardinalidad (59 y 60 valores); el resto
son binarias o de pocas categorias, sin riesgo de explosion dimensional.

## 4. Datos faltantes

Se separan dos bloques: las columnas `id_*` y `Device*`, cuyo faltante viene
del left join con `train_identity` (estructural, no error de captura), y el
resto de columnas, donde hay que evaluar si el mecanismo es MCAR, MAR o
MNAR antes de decidir como tratarlas.

In [ ]:
missing_pct = (df.isna().mean() * 100).round(2)
missing_id_device = missing_pct[missing_pct.index.isin(id_device_cols)].sort_values(ascending=False)
missing_resto = missing_pct[~missing_pct.index.isin(id_device_cols)].sort_values(ascending=False)
pct_sin_identity = df[id_device_cols[0]].isna().mean() * 100

barh(missing_id_device.head(20), "Porcentaje de faltantes: bloque id_/Device (20 peores)",
     "% faltante", color="#DD8452")
print(f"{pct_sin_identity:.1f}% de las filas no tienen fila de identity asociada. "
      "Ese numero por si solo explica la mayor parte del missing de este bloque.")

In [ ]:
barh(missing_resto.head(30), "Porcentaje de faltantes: resto de columnas (30 peores)",
     "% faltante", figsize=(9, 8))

El faltante de `id_*`/`Device*` esta condicionado casi por completo a si
la transaccion tiene fila de identity: es MAR respecto a esa variable
observable, y estructural por diseño del left join, no un error de captura.
No corresponde imputarlo con la media; corresponde marcar un indicador
`has_identity` y tratar la ausencia como su propia categoria.

Fuera de ese bloque, columnas como `dist2`, `D7`, `D12`, `D13`, `D14` superan
87% de faltante, y un grupo grande de columnas `V` comparte exactamente el
mismo porcentaje de faltante. Esa coincidencia exacta es la primera pista de
que el missing viene en bloques, no columna por columna.

In [ ]:
v_sample_cols = v_cols[::6][:60]
sample_rows = df[v_sample_cols].sample(n=300, random_state=42).sort_index()
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(sample_rows.isna(), cbar=False, ax=ax, cmap=["#4C72B0", "#DD8452"])
ax.set_title("Patron de faltantes en columnas V (muestra de 300 filas, 1 de cada 6 columnas V)")
ax.set_xlabel("columnas V (submuestreadas)")
ax.set_ylabel("filas (muestra)")
plt.tight_layout()
plt.show()

v_missing_counts = df[v_cols].isna().sum()
n_firmas = v_missing_counts.value_counts().shape[0]
print(f"Las 339 columnas V colapsan en solo {n_firmas} porcentajes de faltante distintos, "
      "es decir, grupos enteros de columnas faltan juntos en las mismas filas.")

In [ ]:
miss_by_product = df.groupby("ProductCD", observed=True)[v_cols[0]].apply(lambda s: s.isna().mean() * 100)
fig, ax = plt.subplots(figsize=(6, 4))
miss_by_product.sort_values().plot(kind="bar", ax=ax, color="#55A868")
ax.set_title(f"Porcentaje de faltantes de {v_cols[0]} segun ProductCD")
ax.set_ylabel("% faltante")
plt.tight_layout()
plt.show()
print(f"{v_cols[0]} pasa de casi 100% de faltante en varios ProductCD a un porcentaje "
      "mucho menor en otros: el faltante depende de una variable observada.")

El heatmap muestra bloques visibles de columnas que faltan juntas en las
mismas filas, las 339 columnas V colapsan a un puñado de porcentajes de
faltante compartidos, y el faltante de `V1` depende claramente de
`ProductCD`. Los tres hechos apuntan en la misma direccion: el mecanismo es
MAR, condicionado a variables observadas como `ProductCD`, consistente con
que Vesta describe las columnas V como features de ingenieria calculadas en
bloques relacionados. La recomendacion practica es tratar cada bloque como
una unidad al imputar, y agregar indicadores de "bloque ausente" como
feature adicional en vez de solo rellenar valores.

## 5. Consistencia y valores atipicos

Se revisa si `TransactionAmt` tiene inconsistencias de formato, y se comparan
dos metodos de deteccion de outliers, IQR y z-score, tanto en `TransactionAmt`
como en las columnas `D`, sin eliminar nada todavia: primero hay que ver si
los valores extremos son error o señal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["TransactionAmt"].clip(upper=df["TransactionAmt"].quantile(0.99)).hist(bins=80, ax=axes[0], color="#4C72B0")
axes[0].set_title("TransactionAmt (recortado en percentil 99 solo para graficar)")
np.log1p(df["TransactionAmt"]).hist(bins=80, ax=axes[1], color="#55A868")
axes[1].set_title("log1p(TransactionAmt)")
plt.tight_layout()
plt.show()

In [ ]:
amt = df["TransactionAmt"].dropna()
q1, q3 = amt.quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
mask_iqr = (amt < lo) | (amt > hi)
pct_iqr = round(mask_iqr.mean() * 100, 2)
pct_z = round((np.abs(stats.zscore(amt.astype("float64"))) > 3).mean() * 100, 2)
tasa_fraude_outliers = df.loc[amt[mask_iqr].index, "isFraud"].mean() * 100
tasa_fraude_global = df["isFraud"].mean() * 100

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["IQR (1.5x)", "Z-score (> 3)"], [pct_iqr, pct_z], color=["#DD8452", "#C44E52"])
ax.set_ylabel("% de filas marcadas como outlier")
ax.set_title("TransactionAmt: IQR frente a z-score")
plt.tight_layout()
plt.show()

frac = (df["TransactionAmt"] * 100).round(4) % 1
pct_mas_2_dec = round((frac.abs() > 1e-6).mean() * 100, 3)
print(f"IQR marca {pct_iqr}% de las filas como outlier; z-score solo {pct_z}%. "
      f"La diferencia se explica por la asimetria fuerte de la distribucion (skew = {stats.skew(amt):.1f}).")
print(f"La tasa de fraude dentro de los outliers de IQR es {tasa_fraude_outliers:.2f}%, "
      f"frente a {tasa_fraude_global:.2f}% en el dataset completo.")
print(f"{pct_mas_2_dec}% de las filas tienen mas de dos decimales en TransactionAmt, "
      "compatible con conversion de moneda extranjera.")

In [ ]:
d_outlier_pct = {}
for c in d_cols:
    s = df[c].dropna()
    if len(s) < 1000:
        continue
    q1c, q3c = s.quantile([0.25, 0.75])
    iqrc = q3c - q1c
    if iqrc == 0:
        continue
    loc, hic = q1c - 1.5 * iqrc, q3c + 1.5 * iqrc
    d_outlier_pct[c] = round(float(((s < loc) | (s > hic)).mean() * 100), 2)
barh(pd.Series(d_outlier_pct), "Outliers por IQR en columnas D", "% de filas marcadas", color="#C44E52")

Ni en `TransactionAmt` ni en las columnas D aparecen valores imposibles
como montos o dias negativos, asi que no hay evidencia de error de captura.
La cola pesada es consistente con comportamiento real: compras de monto alto,
o cuentas que se reactivan despues de mucho tiempo, lo que eleva ciertas
columnas D. La tasa de fraude dentro de los outliers de `TransactionAmt` es
mayor que la tasa global, asi que estos valores extremos llevan algo de
señal real. La decision es conservarlos, y agregar `amt_multi_decimal` como
feature en vez de redondear los montos con mas de dos decimales.

## 6. Riesgo de fuga de datos

Esta es la seccion mas sensible del EDA porque un error aqui infla las
metricas de validacion sin que se note hasta produccion. Se revisan cuatro
frentes: correlacion cruda con el target, orden temporal, agrupacion de
entidad (mismo cliente en train y validacion), y el hecho de no haber corrido
todavia ningun preprocesamiento sobre el dataset completo.

In [ ]:
numeric_cols = [c for c in df.select_dtypes(include=["float32", "float64", "int32", "int8"]).columns
                if c not in ("isFraud", "TransactionID")]
corrs = df[numeric_cols].corrwith(df["isFraud"].astype("float32"))
top20 = corrs.reindex(corrs.abs().sort_values(ascending=False).head(20).index)
barh(top20, "Las 20 variables mas correlacionadas con isFraud", "correlacion (con signo)", color="#8172B2")

corr_dt = corrs.get("TransactionDT", float("nan"))
print(f"Correlacion cruda de TransactionDT con isFraud: {corr_dt:.4f}")
print(f"Correlacion maxima observada en todo el dataset: {corrs.abs().max():.3f}")

Ninguna variable llega a una correlacion cercana a 1.0 con el target, asi
que no hay evidencia de una variable que sea una copia disfrazada de
`isFraud`. La correlacion maxima, alrededor de 0.38 en columnas V, es alta
para un problema anonimizado pero razonable para features de ingenieria
relacionadas con fraude. `TransactionDT` en crudo casi no correlaciona con el
target: el riesgo de usarlo no es de correlacion espuria, es de orden. Un
split aleatorio dejaria que el modelo vea el futuro durante el entrenamiento.

In [ ]:
txn_day = (df["TransactionDT"] // (24 * 3600)).astype("int32")
uid_df = df[["card1", "card2", "card3", "card5", "addr1", "addr2"]].copy()
uid_df["_d1_anchor"] = (df["D1"] - txn_day).round(1)
uid = uid_df.groupby(list(uid_df.columns), observed=True, dropna=False).ngroup()
uid_counts = uid.value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
uid_counts.clip(upper=20).value_counts().sort_index().plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_title("Transacciones por cliente candidato (recortado en 20 o mas)")
ax.set_xlabel("transacciones del mismo cliente candidato")
ax.set_ylabel("cantidad de clientes candidatos")
plt.tight_layout()
plt.show()

pct_repetida = round(float(uid_counts[uid_counts > 1].sum() / len(df) * 100), 2)
print("Cliente candidato = card1 + card2 + card3 + card5 + addr1 + addr2 + (D1 ajustado por dia).")
print(f"{len(uid_counts):,} clientes candidatos distintos sobre {len(df):,} filas.")
print(f"{pct_repetida}% de todas las filas pertenecen a un cliente candidato que aparece mas de una vez.")

In [ ]:
cutoff_dt = df["TransactionDT"].quantile(0.80)
lado = pd.Series(np.where(df["TransactionDT"] <= cutoff_dt, "train", "val"), index=df.index)
cruza = lado.groupby(uid).nunique() == 2
filas_afectadas = uid.isin(cruza[cruza].index)
pct_afectadas = round(float(filas_afectadas.mean() * 100), 2)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["No cruza el corte", "Cruza el corte"], [100 - pct_afectadas, pct_afectadas],
       color=["#55A868", "#C44E52"])
ax.set_ylabel("% de filas del dataset")
ax.set_title("Filas cuyo cliente candidato aparece a ambos lados de un corte temporal de prueba")
plt.tight_layout()
plt.show()
print(f"Usando un corte de prueba en el percentil 80 de TransactionDT, "
      f"{int(cruza.sum()):,} clientes candidatos aparecen a ambos lados del corte, "
      f"lo que afecta al {pct_afectadas}% de todas las filas.")

Este es el hallazgo mas importante del EDA. Si el split train/validacion
se hace solo por fecha, sin agrupar por cliente, alrededor de 30% de las
filas de validacion corresponden a un cliente que el modelo ya vio durante
el entrenamiento, y el PR-AUC medido en validacion quedaria inflado de forma
artificial. El split debe combinar dos condiciones a la vez: ser
estrictamente cronologico, y agrupar por esta identidad de cliente candidata
(o una version validada con variantes de `D4`, `D10`, `D15`) para que ningun
cliente cruce el corte.

Sobre el cuarto frente, el de preprocesamiento: en este EDA ninguna
estadistica, frecuencia o escalado se calcula todavia sobre el dataset
completo; ese calculo debe hacerse solo con la ventana de entrenamiento
cuando se construya el pipeline de features, y aplicarse despues a
validacion, nunca al reves.

## 7. Redundancia y variables ruidosas

Se buscan tres cosas: columnas con varianza casi nula, multicolinealidad
entre las columnas V y entre las columnas C, y columnas de cardinalidad muy
alta que puedan funcionar como identificador de cliente disfrazado.

In [ ]:
near_zero = []
for c in df.columns:
    if c in ("TransactionID", "isFraud"):
        continue
    vc = df[c].value_counts(dropna=True, normalize=True)
    if len(vc) > 0 and vc.iloc[0] > 0.99:
        near_zero.append((c, float(vc.iloc[0] * 100)))
nz_series = pd.Series(dict(near_zero)).sort_values(ascending=False)
barh(nz_series.head(25), "25 columnas con mayor concentracion en un solo valor",
     "% de filas con el valor dominante", color="#937860")
print(f"{len(nz_series)} columnas tienen mas de 99% de sus valores concentrados en uno solo.")

In [ ]:
Vmat = df[v_cols].to_numpy(dtype=np.float32)
col_mean = np.nanmean(Vmat, axis=0)
nan_mask = np.isnan(Vmat)
Vmat[nan_mask] = np.take(col_mean, np.where(nan_mask)[1])
Vmat -= Vmat.mean(axis=0)
col_std = Vmat.std(axis=0)
col_std[col_std == 0] = 1e-9
Vmat /= col_std
v_corr = (Vmat.T @ Vmat) / Vmat.shape[0]
v_corr_abs = np.abs(v_corr.copy())
np.fill_diagonal(v_corr_abs, 0)
pares_altos = int((v_corr_abs > 0.9).sum() / 2)
pares_posibles = len(v_cols) * (len(v_cols) - 1) // 2
redundancia = pd.Series((v_corr_abs > 0.9).sum(axis=0), index=v_cols).sort_values(ascending=False)

top_redundant = redundancia.head(35).index.tolist()
idx = [v_cols.index(c) for c in top_redundant]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(v_corr[np.ix_(idx, idx)], xticklabels=top_redundant, yticklabels=top_redundant,
            cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax, cbar_kws={"label": "correlacion"})
ax.set_title("Correlacion entre las 35 columnas V mas redundantes")
plt.tight_layout()
plt.show()
print(f"{pares_altos:,} pares de columnas V tienen correlacion mayor a 0.9, "
      f"de {pares_posibles:,} pares posibles ({pares_altos / pares_posibles * 100:.1f}%).")
del Vmat
gc.collect()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[c_cols].corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlacion entre las columnas C")
plt.tight_layout()
plt.show()

La correlacion entre columnas V con valor mayor a 0.9 solo alcanza a un
porcentaje pequeño del total de pares posibles, pero se concentra en un
subgrupo compacto de columnas altamente redundantes entre si, visible en el
heatmap. Esas columnas son buenas candidatas para reducir via PCA o
clustering, quedandose con un representante por grupo, no para eliminar una
por una a mano. Entre las columnas C, el grupo `C1, C2, C4, C6, C7, C8, C10,
C11, C12, C14` esta casi perfectamente correlacionado entre si y podria
colapsarse en uno o dos componentes; `C3`, `C5`, `C9` y `C13` se comportan
distinto y conviene conservarlas por separado.

In [ ]:
alta_card = {c: df[c].nunique(dropna=True) for c in ["card1", "card2", "card3", "card5", "D1", "D2", "D15"]}
fig, ax = plt.subplots(figsize=(7, 4))
pd.Series(alta_card).sort_values().plot(kind="barh", ax=ax, color="#8C8C8C", logx=True)
ax.set_title("Cardinalidad en escala logaritmica: posibles identificadores de cliente")
ax.set_xlabel("cantidad de valores unicos (escala log)")
plt.tight_layout()
plt.show()
print(f"card1 tiene {alta_card['card1']:,} valores unicos, "
      f"equivalente a {alta_card['card1'] / len(df) * 100:.1f}% del total de filas.")

`card1` en particular tiene cardinalidad alta, en linea con lo esperado si
funciona como proxy de tarjeta o cliente, algo ya sugerido por su presencia
en el identificador candidato de la seccion anterior. No conviene tratarlo
como una categorica cruda con one-hot encoding; conviene usar frequency o
target encoding, ajustado exclusivamente con la ventana de entrenamiento.

## 8. Balance de clases y variacion temporal

Se confirma el desbalance de clases y se revisa si la tasa de fraude y el
monto promedio cambian a lo largo del tiempo, como evidencia preliminar de
concept drift antes de entrenar cualquier modelo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
dist = df["isFraud"].value_counts(normalize=True) * 100
axes[0].bar(["Legitima (0)", "Fraude (1)"], dist.reindex([0, 1]), color=["#55A868", "#C44E52"])
axes[0].set_ylabel("%")
axes[0].set_title("Distribucion de la variable objetivo")
for i, v in enumerate(dist.reindex([0, 1])):
    axes[0].text(i, v + 1, f"{v:.1f}%", ha="center")

d_shape = pd.Series({c: stats.skew(df[c].dropna()) for c in d_cols if df[c].dropna().shape[0] > 1000})
d_shape.sort_values().plot(kind="barh", ax=axes[1], color="#4C72B0")
axes[1].set_title("Asimetria (skew) de cada columna D")
plt.tight_layout()
plt.show()
print(f"Distribucion de isFraud: {dist[0]:.1f}% legitimas, {dist[1]:.1f}% fraude.")

In [ ]:
bin_temporal = pd.qcut(df["TransactionDT"], 10, labels=False, duplicates="drop")
shift = df.groupby(bin_temporal, observed=True).agg(
    isFraud_rate=("isFraud", "mean"), TransactionAmt_mean=("TransactionAmt", "mean"))
shift["isFraud_rate"] *= 100

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
shift["isFraud_rate"].plot(marker="o", ax=axes[0], color="#C44E52")
axes[0].set_title("Tasa de fraude por decil de TransactionDT")
axes[0].set_xlabel("decil temporal, 0 es el mas antiguo")
shift["TransactionAmt_mean"].plot(marker="o", ax=axes[1], color="#4C72B0")
axes[1].set_title("TransactionAmt promedio por decil de TransactionDT")
axes[1].set_xlabel("decil temporal")
plt.tight_layout()
plt.show()
print(f"La tasa de fraude oscila entre {shift['isFraud_rate'].min():.2f}% "
      f"y {shift['isFraud_rate'].max():.2f}% segun el decil temporal.")

Con un desbalance de aproximadamente 96.5% contra 3.5%, un modelo que
siempre prediga "legitima" tendria una accuracy engañosamente alta; de ahi
que el proyecto use PR-AUC como metrica principal, y recall, precision y F1
como complementarias. La tasa de fraude no es estable en el tiempo: llega a
mas que duplicarse entre el decil mas bajo y el mas alto, lo que confirma que
el problema tiene un componente no estacionario real y no solo teorico. Esto
refuerza la necesidad de un split cronologico y de monitorear drift una vez
el modelo este en produccion.

## 9. Equidad y limitaciones

Las columnas `id_*`, `card*`, `addr*` y los bloques V, C, D estan
anonimizadas y Vesta no documenta su significado real. Por esa razon, este
EDA no intenta inferir caracteristicas demograficas como edad, genero o
ubicacion especifica a partir de ellas. El dataset no ofrece columnas
interpretables que permitan una evaluacion de equidad valida entre grupos
protegidos, asi que esto se reporta como una limitacion del sistema, en vez
de forzar una segmentacion sobre variables sin significado conocido.

## 10. Conclusiones del EDA

Variables a descartar o tratar con cuidado antes de modelar:

- `TransactionID`, unicamente llave de union, no aporta como feature.
- `TransactionDT` en crudo, solo debe usarse para ordenar y particionar, no
  como numero que se entrega directo al modelo; sirve para derivar hora del
  dia o dia de la semana.
- Alrededor de 30 columnas con varianza casi nula (`addr2`, `C3`, `M1`, un
  grupo de columnas V, `id_04`, `id_27`), candidatas a eliminar o binarizar,
  verificando antes la tasa de fraude en el valor minoritario de cada una.
- El grupo de columnas V mas redundantes entre si (ver el heatmap de la
  seccion 7), candidato a reduccion por PCA o clustering.
- El grupo de columnas C casi perfectamente correlacionado
  (`C1, C2, C4, C6, C7, C8, C10, C11, C12, C14`), candidato a colapsar en uno
  o dos componentes.

Variables que no son ruido pero requieren tratamiento especial:

- `card1`, `card2`, `card3`, `card5`, `D1`, `D2`, `D15`: alta cardinalidad,
  riesgo de actuar como identificador de cliente disfrazado. Requieren
  frequency o target encoding ajustado solo con la ventana de entrenamiento.
- `id_02`, con mas de cien mil valores unicos, requiere investigarse antes de
  usarse en crudo.

El hallazgo con mayor impacto en la validez de las metricas es el de la
seccion 6: alrededor de 30% de las filas comparten cliente candidato entre
ambos lados de un corte temporal de prueba. El split final debe agrupar por
esa identidad de cliente ademas de ser cronologico.

Quedan dos decisiones de negocio que este EDA no puede asumir por su cuenta:
la fecha exacta de corte entre entrenamiento y validacion, y el margen de
dias que debe dejarse entre el fin de entrenamiento y el inicio de
validacion para simular el retraso realista en la confirmacion de fraude. La
seccion 11 documenta los supuestos concretos que este equipo adopto para
poder avanzar, precisamente porque nadie los confirmo todavia, y deja claro
que son parametros reemplazables, no verdades del dataset.

## 11. Del EDA al pipeline: decisiones y supuestos

Esta seccion traduce las conclusiones del EDA en decisiones de ingenieria
concretas, y deja explicitos los supuestos de negocio que el EDA identifico
como pendientes pero que nadie del equipo o del curso confirmo todavia. Se
adoptan valores razonables y se documentan aqui mismo para que sean faciles
de reemplazar si se recibe una instruccion distinta.

Decisiones de features que se ejecutan en la seccion 13, heredadas
directamente de la seccion 10 del EDA:

- Se descarta `TransactionID` (solo era llave de union) y `TransactionDT` en
  crudo (se usa unicamente para ordenar y para derivar hora del dia y dia de
  la semana).
- Se descartan las columnas con mas de 99% de su masa en un solo valor
  (varianza casi nula), recalculadas sobre la ventana de entrenamiento, no
  sobre el dataset completo.
- Entre las columnas V con correlacion mayor a 0.95 entre si, se conserva
  solo una por grupo redundante, tambien ajustado sobre la ventana de
  entrenamiento.
- `card1`, `card2`, `card3`, `addr1` y el identificador de cliente candidato
  (`uid`) se transforman con frequency encoding ajustado en entrenamiento, no
  con one-hot.
- Se agregan estadisticos por `uid` (monto promedio, desviacion, `D1`
  promedio) calculados solo con la ventana de entrenamiento y mapeados hacia
  adelante, mas un indicador `uid_is_new` para clientes que no existian en
  entrenamiento, y el ratio `amt_to_uid_mean_ratio` entre el monto de cada
  transaccion y el promedio historico de ese cliente.
- Se agrega una segunda variante de identidad de cliente (`uid2`, con ancla
  en `D15` en vez de `D1`) como frequency encoding adicional, cerrando el
  punto que el EDA dejo pendiente de validar otras anclas temporales
  (`D4`, `D10`, `D15`).
- Se agrega `time_since_prev_uid`, el tiempo transcurrido desde la
  transaccion anterior del mismo `uid`: una rafaga de transacciones muy
  seguidas del mismo cliente candidato es una señal clasica de fraude, y el
  calculo es causal (solo mira hacia atras en el tiempo de cada fila).

Supuestos de negocio que el EDA dejo pendientes y que este pipeline resuelve
de forma transparente:

- Corte de entrenamiento: percentil 60 de `TransactionDT`. Gap de latencia de
  etiqueta: hasta el percentil 63 (aproximadamente 5 a 6 dias, ilustrativo,
  no un dato confirmado del proceso real de confirmacion de fraude).
  Validacion: percentil 63 a 75. Prueba: percentil 75 en adelante, dividido
  en 5 ventanas para el monitoreo temporal de las secciones 18 y 19.
- Costos de la politica de decision de la seccion 17: se ancla el costo de un
  fraude no detectado al monto de la transaccion multiplicado por 4.41, la
  cifra que LexisNexis Risk Solutions (2024) reporta como costo total por
  cada dolar perdido directamente por fraude, citada en el planteamiento del
  proyecto. El costo de friccion por bloquear una transaccion legitima y el
  costo de una revision manual se fijan en valores ilustrativos pequeños y
  se documentan en la seccion 17 mismo.
- Capacidad de revision manual: se asume un maximo de 5% de las transacciones
  de cada ventana, tambien ilustrativo.

Estos numeros son parametros del sistema, no hallazgos del dataset:
cualquier cifra real de negocio que el curso o un caso real entregue despues
reemplaza directamente a estos supuestos sin cambiar el resto del pipeline.

## 12. Split temporal con gap de latencia de etiqueta

Se construyen cuatro particiones sobre `TransactionDT`, nunca por muestreo
aleatorio: entrenamiento, un hueco (gap) sin uso que simula el retraso real
en la confirmacion de fraude, validacion, y prueba. La prueba se reserva
integra para las secciones 15 en adelante y se subdivide en ventanas
recientes para el monitoreo temporal.

In [ ]:
train_end = df["TransactionDT"].quantile(0.60)
gap_end = df["TransactionDT"].quantile(0.63)
val_end = df["TransactionDT"].quantile(0.75)

train_mask = df["TransactionDT"] <= train_end
gap_mask = (df["TransactionDT"] > train_end) & (df["TransactionDT"] <= gap_end)
val_mask = (df["TransactionDT"] > gap_end) & (df["TransactionDT"] <= val_end)
test_mask = df["TransactionDT"] > val_end

gap_dias = (gap_end - train_end) / (3600 * 24)

resumen_split = pd.DataFrame({
    "particion": ["entrenamiento", "gap (no se usa)", "validacion", "prueba"],
    "n_filas": [int(train_mask.sum()), int(gap_mask.sum()), int(val_mask.sum()), int(test_mask.sum())],
    "tasa_fraude_pct": [round(float(df.loc[m, "isFraud"].mean() * 100), 2)
                         for m in [train_mask, gap_mask, val_mask, test_mask]],
})
print(resumen_split.to_string(index=False))
print(f"Gap de latencia de etiqueta: {gap_dias:.1f} dias.")

fig, ax = plt.subplots(figsize=(9, 2.2))
for nombre, m, color in [("train", train_mask, "#4C72B0"), ("gap", gap_mask, "#8C8C8C"),
                          ("val", val_mask, "#DD8452"), ("test", test_mask, "#C44E52")]:
    ax.scatter(df.loc[m, "TransactionDT"], [0] * int(m.sum()), s=1, color=color, label=nombre)
ax.set_yticks([])
ax.set_xlabel("TransactionDT")
ax.set_title("Particion temporal: entrenamiento, gap, validacion, prueba")
ax.legend(markerscale=8, loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=4)
plt.tight_layout()
plt.show()
save_progress("split_temporal")

Las cuatro particiones son bloques contiguos en el tiempo, sin traslape:
el modelo nunca entrena con datos posteriores a los que usa para validar o
probar. La tasa de fraude cambia entre particiones, coherente con el drift
temporal ya visto en la seccion 8, y es precisamente lo que las secciones 18
y 19 van a monitorear y a intentar mitigar.

## 13. Ingenieria de variables

Todo estadistico usado para transformar una columna (frecuencias,
agregados por cliente, categorias validas) se ajusta unicamente con las filas
de entrenamiento y despues se aplica hacia adelante sobre gap, validacion y
prueba. Es la misma regla que la seccion 6 identifico como condicion para
evitar fuga de preprocesamiento.

In [ ]:
nzv_cols = []
for c in df.columns:
    if c in ("TransactionID", "isFraud", "TransactionDT"):
        continue
    vc = df.loc[train_mask, c].value_counts(normalize=True, dropna=True)
    if len(vc) > 0 and vc.iloc[0] > 0.99:
        nzv_cols.append(c)

Vmat = df.loc[train_mask, v_cols].to_numpy(dtype=np.float32)
col_mean = np.nanmean(Vmat, axis=0)
nan_mask = np.isnan(Vmat)
Vmat[nan_mask] = np.take(col_mean, np.where(nan_mask)[1])
Vmat -= Vmat.mean(axis=0)
col_std = Vmat.std(axis=0)
col_std[col_std == 0] = 1e-9
Vmat /= col_std
v_corr_train = (Vmat.T @ Vmat) / Vmat.shape[0]
del Vmat
gc.collect()

idx_map = {c: i for i, c in enumerate(v_cols)}
kept_v, dropped_v = [], []
for c in v_cols:
    if c in nzv_cols:
        dropped_v.append(c)
        continue
    redundante = any(abs(v_corr_train[idx_map[c], idx_map[k]]) > 0.95 for k in kept_v)
    (dropped_v if redundante else kept_v).append(c)

print(f"Columnas con varianza casi nula (ajustado en train): {len(nzv_cols)}")
print(f"Columnas V conservadas: {len(kept_v)} de {len(v_cols)} "
      f"({len(dropped_v)} descartadas por redundancia o varianza casi nula).")

In [ ]:
# ponytail: los agregados por uid incluyen la propia fila cuando esta en train
# (leve fuga in-sample); si el gap entre train y val se reduce, conviene pasar
# a un agregado leave-one-out.
txn_day = (df["TransactionDT"] // 86400).astype("int32")
uid_df = df[["card1", "card2", "card3", "card5", "addr1", "addr2"]].copy()
uid_df["_d1_anchor"] = (df["D1"] - txn_day).round(1)
df["uid"] = uid_df.groupby(list(uid_df.columns), observed=True, dropna=False).ngroup()

freq_cols = ["card1", "card2", "card3", "addr1", "uid"]
for c in freq_cols:
    freq_map = df.loc[train_mask, c].value_counts()
    df[f"{c}_freq"] = df[c].map(freq_map).fillna(0).astype("float32")

uid_stats_train = df.loc[train_mask].groupby("uid").agg(
    uid_amt_mean=("TransactionAmt", "mean"),
    uid_amt_std=("TransactionAmt", "std"),
    uid_d1_mean=("D1", "mean"),
)
uids_en_train = set(uid_stats_train.index)
df = df.merge(uid_stats_train, on="uid", how="left")
df["uid_is_new"] = (~df["uid"].isin(uids_en_train)).astype("int8")
df["amt_to_uid_mean_ratio"] = (df["TransactionAmt"] / df["uid_amt_mean"].replace(0, np.nan)).astype("float32")

# Segunda variante de uid, ancla en D15 en vez de D1: la seccion 6 del EDA
# quedo pendiente de probar variantes con D4/D10/D15; esta es esa validacion.
uid2_df = df[["card1", "card2", "addr1"]].copy()
uid2_df["_d15_anchor"] = (df["D15"] - txn_day).round(1)
df["uid2"] = uid2_df.groupby(list(uid2_df.columns), observed=True, dropna=False).ngroup()
freq_map_uid2 = df.loc[train_mask, "uid2"].value_counts()
df["uid2_freq"] = df["uid2"].map(freq_map_uid2).fillna(0).astype("float32")

# Tiempo desde la transaccion anterior del mismo cliente candidato: la rafaga
# de transacciones muy seguidas del mismo uid es una señal clasica de fraude
# (y es calculable sin fuga, porque cada fila solo mira hacia atras en el tiempo).
df = df.sort_values("TransactionDT")
df["time_since_prev_uid"] = df.groupby("uid")["TransactionDT"].diff().astype("float32")
df["time_since_prev_uid"] = df["time_since_prev_uid"].fillna(-1)
df = df.sort_index()

df["hour_of_day"] = ((df["TransactionDT"] // 3600) % 24).astype("int8")
df["day_of_week"] = ((df["TransactionDT"] // 86400) % 7).astype("int8")
df["amt_log1p"] = np.log1p(df["TransactionAmt"]).astype("float32")
df["amt_multi_decimal"] = (((df["TransactionAmt"] * 100).round(4) % 1).abs() > 1e-6).astype("int8")
df["has_identity"] = df[id_device_cols[0]].notna().astype("int8")

print(f"{df['uid'].nunique():,} clientes candidatos (uid, ancla D1) construidos sobre el dataset completo, "
      f"{len(uids_en_train):,} de ellos presentes en entrenamiento.")
print(f"{df['uid2'].nunique():,} clientes candidatos (uid2, ancla D15) como segunda variante de identidad.")

In [ ]:
cat_cols = [c for c in df.select_dtypes(include="category").columns]
label_maps = {}
for c in cat_cols:
    valores_train = df.loc[train_mask, c].astype("string").fillna("__missing__").unique()
    mapping = {v: i for i, v in enumerate(valores_train)}
    label_maps[c] = mapping
    codigos = df[c].astype("string").fillna("__missing__").map(mapping)
    df[f"{c}_enc"] = codigos.fillna(-1).astype("int32")

print(f"{len(cat_cols)} columnas categoricas codificadas por label encoding "
      "(-1 marca una categoria que no existia en entrenamiento).")

In [ ]:
base_numeric = [c for c in (["TransactionAmt"] + c_cols + d_cols) if c not in nzv_cols]
freq_feats = [f"{c}_freq" for c in freq_cols] + ["uid2_freq"]
uid_feats = ["uid_amt_mean", "uid_amt_std", "uid_d1_mean", "uid_is_new", "amt_to_uid_mean_ratio"]
cat_enc_feats = [f"{c}_enc" for c in cat_cols]
extra_feats = ["hour_of_day", "day_of_week", "amt_log1p", "amt_multi_decimal", "has_identity",
               "time_since_prev_uid"]

feature_cols = base_numeric + kept_v + freq_feats + uid_feats + cat_enc_feats + extra_feats
X = df[feature_cols]
y = df["isFraud"].astype("int8")

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print(f"Total de features: {len(feature_cols)}")
print(f"Entrenamiento: {X_train.shape}   Validacion: {X_val.shape}   Prueba: {X_test.shape}")

save_artifact({"feature_cols": feature_cols, "label_maps": label_maps, "kept_v": kept_v,
               "nzv_cols": nzv_cols, "freq_cols": freq_cols}, "feature_engineering.joblib")
save_progress("features_listas")

El resultado es una tabla de features numerica y estable: todas las
decisiones de que descartar, agrupar o codificar quedan fijadas por la
ventana de entrenamiento y se aplican igual hacia adelante. Las columnas
`D` y `C` se dejan con sus valores faltantes intactos (sin imputar con la
media), porque LightGBM, XGBoost y CatBoost manejan `NaN` nativamente y una
imputacion ciega borraria la señal de "faltante" que la seccion 4 del EDA
identifico como informativa.

## 14. Manejo del desbalance de clases

El dataset tiene aproximadamente 96.5% de transacciones legitimas contra
3.5% de fraude. La decision es no usar sobremuestreo sintetico tipo SMOTE:
generar filas sinteticas interpolando entre columnas `V`/`C`/`D` anonimizadas
no tiene una geometria interpretable (son features de ingenieria de Vesta, no
mediciones fisicas continuas), y ademas rompe la logica temporal del
problema, porque una fila sintetica no es una transaccion real que ocurrio en
un momento determinado. En su lugar, el desbalance se maneja reponderando la
funcion de perdida de cada modelo (`scale_pos_weight`, o `class_weight` en la
regresion logistica), calculado solo con la ventana de entrenamiento, y
evaluando con PR-AUC en vez de accuracy. La seccion 17 anade una capa
adicional para el desbalance de costos de negocio, no solo de clases.

In [ ]:
n_pos = int(y_train.sum())
n_neg = int((y_train == 0).sum())
scale_pos_weight = n_neg / n_pos
print(f"Entrenamiento: {n_pos:,} fraudes de {len(y_train):,} filas ({n_pos / len(y_train) * 100:.2f}%).")
print(f"scale_pos_weight = {scale_pos_weight:.1f}")

## 15. Modelos, metricas y conclusiones (Objetivo 1)

Se comparan cuatro modelos bajo el mismo split temporal: una regresion
logistica como referencia tradicional, y LightGBM, XGBoost y CatBoost como
modelos avanzados de arbol de gradiente. El diseño de features
(identificador de cliente candidato, frequency encoding, agregados por
cliente, reduccion de columnas V redundantes) esta inspirado en el enfoque
que uso el equipo ganador de esta competencia (FraudSquad, primer lugar), sin
replicar su solucion exacta: aqui se usan menos modelos, sin ensamble final
ni adversarial validation, priorizando un pipeline que se pueda explicar y
reproducir de punta a punta. Los cuatro modelos se comparan por PR-AUC en
validacion; el mejor se vuelve a evaluar en prueba, una ventana de tiempo que
ningun modelo vio durante el ajuste. Al final de la seccion se agrega, como
benchmark secundario de baja prioridad, un MLP (red neuronal densa) entrenado
sobre las mismas features imputadas y escaladas.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                              precision_recall_curve, precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

lr_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=200, class_weight="balanced", n_jobs=-1)),
])
lr_pipe.fit(X_train, y_train)
pr_auc_lr = average_precision_score(y_val, lr_pipe.predict_proba(X_val)[:, 1])
print(f"Regresion logistica (linea base) — PR-AUC en validacion: {pr_auc_lr:.4f}")

In [ ]:
import lightgbm as lgb

lgb_params = dict(
    objective="binary", n_estimators=4000, learning_rate=0.02, num_leaves=384,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.6, reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight, random_state=42, n_jobs=-1,
)
try:
    lgb_model = lgb.LGBMClassifier(device="gpu", **lgb_params)
    lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="average_precision",
                  callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])
except Exception as exc:
    print(f"LightGBM GPU no disponible en este entorno ({exc}); reintentando en CPU.")
    lgb_model = lgb.LGBMClassifier(device="cpu", **lgb_params)
    lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="average_precision",
                  callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])

pr_auc_lgb = average_precision_score(y_val, lgb_model.predict_proba(X_val)[:, 1])
print(f"LightGBM — PR-AUC en validacion: {pr_auc_lgb:.4f}")

In [ ]:
import xgboost as xgb

xgb_params = dict(
    objective="binary:logistic", eval_metric="aucpr", n_estimators=4000, learning_rate=0.02,
    max_depth=10, min_child_weight=1, subsample=0.8, colsample_bytree=0.6, reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight, tree_method="hist", device="cuda", random_state=42,
    early_stopping_rounds=150,
)
try:
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
except Exception as exc:
    print(f"XGBoost GPU no disponible en este entorno ({exc}); reintentando en CPU.")
    xgb_params["device"] = "cpu"
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

pr_auc_xgb = average_precision_score(y_val, xgb_model.predict_proba(X_val)[:, 1])
print(f"XGBoost — PR-AUC en validacion: {pr_auc_xgb:.4f}")

In [ ]:
from catboost import CatBoostClassifier

cat_params = dict(
    loss_function="Logloss", eval_metric="PRAUC", iterations=4000, learning_rate=0.02,
    depth=9, l2_leaf_reg=3.0, scale_pos_weight=scale_pos_weight, random_seed=42, verbose=False,
    early_stopping_rounds=150,
)
try:
    cat_model = CatBoostClassifier(task_type="GPU", devices="0", **cat_params)
    cat_model.fit(X_train, y_train, eval_set=(X_val, y_val))
except Exception as exc:
    print(f"CatBoost GPU no disponible en este entorno ({exc}); reintentando en CPU.")
    cat_model = CatBoostClassifier(task_type="CPU", **cat_params)
    cat_model.fit(X_train, y_train, eval_set=(X_val, y_val))

pr_auc_cat = average_precision_score(y_val, cat_model.predict_proba(X_val)[:, 1])
print(f"CatBoost — PR-AUC en validacion: {pr_auc_cat:.4f}")

In [ ]:
comparacion = pd.Series({
    "Regresion logistica": pr_auc_lr, "LightGBM": pr_auc_lgb,
    "XGBoost": pr_auc_xgb, "CatBoost": pr_auc_cat,
})
barh(comparacion, "Comparacion de modelos por PR-AUC en validacion", "PR-AUC", figsize=(7, 3.5))

modelos = {"Regresion logistica": lr_pipe, "LightGBM": lgb_model, "XGBoost": xgb_model, "CatBoost": cat_model}
best_name = comparacion.idxmax()
best_model = modelos[best_name]
proba_val = best_model.predict_proba(X_val)[:, 1]
proba_test = best_model.predict_proba(X_test)[:, 1]
pr_auc_test = average_precision_score(y_test, proba_test)

print(f"Mejor modelo en validacion: {best_name} (PR-AUC={comparacion.max():.4f})")
print(f"{best_name} evaluado en prueba (ventana futura nunca vista): PR-AUC={pr_auc_test:.4f}")

save_artifact(best_model, "best_model.joblib")
progress["mejor_modelo"] = best_name
progress["pr_auc_validacion"] = float(comparacion.max())
progress["pr_auc_prueba"] = float(pr_auc_test)
save_progress("modelos_comparados")

In [ ]:
if best_name in ("LightGBM", "XGBoost", "CatBoost"):
    importancias = pd.Series(best_model.feature_importances_, index=feature_cols)
    importancias = importancias.sort_values(ascending=False).head(25)
    barh(importancias, f"25 variables mas importantes segun {best_name}", "importancia")

### Benchmark secundario: MLP

Como comparacion adicional de baja prioridad, se entrena una red neuronal
densa (MLP) sobre las mismas features, pero imputadas y escaladas (a
diferencia de los arboles de gradiente, una red neuronal no maneja `NaN` ni
escalas dispares de forma nativa). A diferencia de los otros cuatro modelos,
`MLPClassifier` de scikit-learn no admite pesos por clase ni por muestra, asi
que este benchmark no corrige el desbalance de clases; se incluye solo como
punto de referencia de un enfoque de deep learning genérico, no como
candidato real a mejor modelo del pipeline.

In [ ]:
from sklearn.neural_network import MLPClassifier

imputer_lr = lr_pipe.named_steps["imputer"]
scaler_lr = lr_pipe.named_steps["scaler"]
X_train_imp = scaler_lr.transform(imputer_lr.transform(X_train))
X_val_imp = scaler_lr.transform(imputer_lr.transform(X_val))
X_test_imp = scaler_lr.transform(imputer_lr.transform(X_test))

mlp_model = MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu", alpha=1e-4,
                           learning_rate_init=1e-3, early_stopping=True, n_iter_no_change=10,
                           max_iter=100, random_state=42)
mlp_model.fit(X_train_imp, y_train)
pr_auc_mlp = average_precision_score(y_val, mlp_model.predict_proba(X_val_imp)[:, 1])
print(f"MLP (benchmark secundario) — PR-AUC en validacion: {pr_auc_mlp:.4f}")

### Bloque de metricas: comparacion completa en prueba

Con los cinco modelos ya entrenados, se arma una tabla unica de metricas
sobre la particion de prueba (nunca vista durante el ajuste), se superponen
sus curvas Precision-Recall en un solo grafico, y se revisa la matriz de
confusion del modelo elegido con el umbral por defecto de 0.5 (el umbral
operativo real se define recien en la seccion 17, con la politica de
costo).

La tabla incluye tanto PR-AUC (la metrica principal de este proyecto) como
ROC-AUC, precisamente para poder comparar contra el leaderboard publico de
la competencia, cuya metrica oficial es ROC-AUC, no PR-AUC. Con clases tan
desbalanceadas (3.5% de fraude) ambas metricas se mueven de forma muy
distinta: ROC-AUC evalua la separacion entre clases sobre toda la poblacion
y tiende a dar valores altos incluso con un modelo mediocre en la clase
minoritaria, porque el 96.5% de negativos domina el calculo. PR-AUC, en
cambio, se concentra en que tan bien se ordena precisamente a la clase
minoritaria, y penaliza con mucha mas dureza los falsos positivos y negativos
sobre fraude. Por eso el proyecto exige PR-AUC como metrica principal: es la
version dificil y honesta del mismo problema. Un ROC-AUC de esta tabla
cercano al de los mejores puestos del leaderboard (que ronda 0.94-0.95) es
la comparacion correcta con esa cifra publica; comparar directamente PR-AUC
contra 0.9459 no es correcto, porque esa cifra del leaderboard nunca fue
PR-AUC.

In [ ]:
modelos_metricas_proba = {
    "Regresion logistica": lr_pipe.predict_proba(X_test)[:, 1],
    "LightGBM": lgb_model.predict_proba(X_test)[:, 1],
    "XGBoost": xgb_model.predict_proba(X_test)[:, 1],
    "CatBoost": cat_model.predict_proba(X_test)[:, 1],
    "MLP (benchmark secundario)": mlp_model.predict_proba(X_test_imp)[:, 1],
}

filas_metricas = []
for nombre, proba in modelos_metricas_proba.items():
    pred_05 = (proba >= 0.5).astype(int)
    filas_metricas.append({
        "modelo": nombre,
        "pr_auc_test": average_precision_score(y_test, proba),
        "roc_auc_test": roc_auc_score(y_test, proba),
        "precision_0.5": precision_score(y_test, pred_05, zero_division=0),
        "recall_0.5": recall_score(y_test, pred_05, zero_division=0),
        "f1_0.5": f1_score(y_test, pred_05, zero_division=0),
    })
metricas_df = pd.DataFrame(filas_metricas).sort_values("pr_auc_test", ascending=False)
print(metricas_df.round(4).to_string(index=False))

progress["metricas_test_todos_los_modelos"] = metricas_df.round(4).to_dict(orient="records")
save_progress("metricas_comparadas")

In [ ]:
colores_modelos = {"Regresion logistica": "#8C8C8C", "LightGBM": "#4C72B0", "XGBoost": "#DD8452",
                    "CatBoost": "#55A868", "MLP (benchmark secundario)": "#C44E52"}

fig, ax = plt.subplots(figsize=(7, 5))
for nombre, proba in modelos_metricas_proba.items():
    prec, rec, _ = precision_recall_curve(y_test, proba)
    estilo = "--" if "secundario" in nombre else "-"
    ax.plot(rec, prec, estilo, label=nombre, color=colores_modelos[nombre])
ax.axhline(y_test.mean(), color="black", linestyle=":", linewidth=1, label="Azar (tasa de fraude)")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Curvas Precision-Recall en prueba, los cinco modelos")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

cm = confusion_matrix(y_test, (proba_test >= 0.5).astype(int))
fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", ax=ax,
            xticklabels=["Predicho legitima", "Predicho fraude"],
            yticklabels=["Real legitima", "Real fraude"])
ax.set_title(f"Matriz de confusion en prueba — {best_name} (umbral 0.5)")
plt.tight_layout()
plt.show()

**Analisis.** La tabla y las curvas Precision-Recall cuentan la misma
historia desde dos angulos: los tres modelos de arbol de gradiente
(LightGBM, XGBoost, CatBoost) dominan de forma consistente a la regresion
logistica y al MLP a lo largo de todo el rango de recall, no solo en un
punto. Eso es coherente con la naturaleza del problema: variables
anonimizadas con relaciones no lineales, cardinalidad alta en algunas
columnas y un desbalance de clases fuerte, exactamente el terreno donde los
arboles de decision con boosting superan a modelos lineales o a una red
densa generica sin ajuste de clase. La matriz de confusion con el umbral por
defecto de 0.5 sirve solo como referencia visual del comportamiento crudo del
modelo elegido; no es la politica operativa final, que se construye en la
seccion 17 a partir del costo esperado en vez de un umbral fijo.

**Conclusiones de esta seccion.** El modelo elegido para el resto del
pipeline es el de mayor PR-AUC en validacion, confirmado en prueba en la
tabla de arriba. La regresion logistica cumple su rol de referencia y
confirma que el problema requiere no linealidad para separar bien ambas
clases. El MLP, sin ajuste de desbalance, se mantiene por debajo de los
arboles de gradiente, lo que valida la decision de la seccion 14 de resolver
el desbalance por reponderacion de la perdida en vez de depender de que el
modelo lo aprenda solo. Estas conclusiones alimentan directamente el resto
del notebook: la verificacion de fuga por cliente y la politica de costo de
las secciones 16 y 17 se calculan siempre sobre el modelo elegido aqui.

## 16. Verificacion empirica de fuga por cliente

La seccion 6 del EDA encontro que alrededor de 30% de las filas comparten
`uid` entre ambos lados de un corte temporal de prueba, y advirtio que esto
podria inflar las metricas de validacion. En vez de forzar un split agrupado
que descartaria datos o distorsionaria el orden temporal (poco realista para
un sistema en produccion, donde un mismo cliente vuelve a transactar), se
hace la prueba directa: se separa el conjunto de prueba entre filas cuyo
`uid` ya aparecio en entrenamiento y filas de un `uid` enteramente nuevo, y se
compara el PR-AUC de cada grupo por separado.

In [ ]:
uid_test_visto = df.loc[test_mask, "uid"].isin(uids_en_train).to_numpy()
y_test_arr = y_test.to_numpy()

pr_auc_visto = average_precision_score(y_test_arr[uid_test_visto], proba_test[uid_test_visto])
pr_auc_nuevo = average_precision_score(y_test_arr[~uid_test_visto], proba_test[~uid_test_visto])

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(["Cliente ya visto\nen entrenamiento", "Cliente nuevo"], [pr_auc_visto, pr_auc_nuevo],
       color=["#C44E52", "#55A868"])
ax.set_ylabel("PR-AUC en prueba")
ax.set_title("PR-AUC segun si el uid ya aparecio en entrenamiento")
plt.tight_layout()
plt.show()
print(f"Filas de prueba con cliente ya visto: {uid_test_visto.mean() * 100:.1f}%, PR-AUC={pr_auc_visto:.4f}")
print(f"Filas de prueba con cliente nuevo: {(~uid_test_visto).mean() * 100:.1f}%, PR-AUC={pr_auc_nuevo:.4f}")

progress["pr_auc_cliente_visto"] = float(pr_auc_visto)
progress["pr_auc_cliente_nuevo"] = float(pr_auc_nuevo)
save_progress("fuga_cliente_verificada")

Si el PR-AUC de clientes ya vistos es notablemente mayor que el de
clientes nuevos, buena parte del desempeño del modelo depende de memorizar
identidad de cliente (via `uid_freq`, `uid_amt_mean`, etc.) mas que de
patrones generalizables de fraude, y ese hallazgo debe reportarse como una
limitacion del desempeño esperado en produccion sobre clientes genuinamente
nuevos. Si ambos PR-AUC son similares, el riesgo de fuga por cliente que
identifico el EDA no se tradujo en una inflacion relevante de la metrica con
este diseño de features, porque todo agregado se ajusto exclusivamente con
la ventana de entrenamiento.

## 17. Politica de decision basada en costo (Objetivo 2)

El riesgo estimado por el modelo se convierte en tres acciones posibles:
aprobar automaticamente, enviar a revision manual, o escalar como fraude.
Dos umbrales, `t_low` y `t_high`, definen los cortes. El costo esperado suma
tres componentes: el costo de un fraude no detectado (aprobado por error),
el costo de friccion de bloquear una transaccion legitima (escalada por
error), y el costo de cada revision manual. Los umbrales se eligen sobre
validacion, minimizando el costo esperado, sujeto a una capacidad maxima de
revision manual.

In [ ]:
cost_fn_mult = 4.41  # LexisNexis (2024): costo total por cada dolar perdido en fraude
cost_fp = 5.0         # friccion ilustrativa de bloquear una transaccion legitima
cost_review = 2.0     # costo ilustrativo de una revision manual
capacidad_revision_pct = 5.0  # tope ilustrativo de transacciones enviadas a revision

amt_val = df.loc[val_mask, "TransactionAmt"].to_numpy()
y_val_arr = y_val.to_numpy()


def costo_esperado(t_low, t_high, proba, y_true, amt):
    aprobar = proba < t_low
    revisar = (proba >= t_low) & (proba < t_high)
    escalar = proba >= t_high
    fn = aprobar & (y_true == 1)
    fp = escalar & (y_true == 0)
    costo = (cost_fn_mult * amt[fn]).sum() + cost_fp * fp.sum() + cost_review * revisar.sum()
    return costo, revisar.mean() * 100, fn, fp, revisar


grid = np.linspace(0.01, 0.95, 60)
resultados = []
for t_low in grid:
    for t_high in grid:
        if t_high <= t_low:
            continue
        costo, pct_rev, *_ = costo_esperado(t_low, t_high, proba_val, y_val_arr, amt_val)
        if pct_rev <= capacidad_revision_pct:
            resultados.append((t_low, t_high, costo, pct_rev))

resultados_df = pd.DataFrame(resultados, columns=["t_low", "t_high", "costo", "pct_revision"])
mejor = resultados_df.loc[resultados_df["costo"].idxmin()]
t_low, t_high = float(mejor["t_low"]), float(mejor["t_high"])
print(f"Umbrales elegidos en validacion: aprobar < {t_low:.3f}, "
      f"revisar en [{t_low:.3f}, {t_high:.3f}), escalar >= {t_high:.3f}")
print(f"Costo esperado en validacion: {mejor['costo']:,.0f}   % enviado a revision: {mejor['pct_revision']:.2f}%")

In [ ]:
amt_test = df.loc[test_mask, "TransactionAmt"].to_numpy()
costo_test, pct_rev_test, fn_test, fp_test, revisar_test = costo_esperado(
    t_low, t_high, proba_test, y_test_arr, amt_test)

costo_baseline, pct_rev_baseline, fn_base, fp_base, _ = costo_esperado(
    0.5, 1.01, proba_test, y_test_arr, amt_test)  # politica de referencia: un solo umbral, sin revision

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.bar(["Politica de referencia\n(umbral unico 0.5)", "Politica de costo\n(2 umbrales)"],
       [costo_baseline, costo_test], color=["#8C8C8C", "#55A868"])
ax.set_ylabel("costo esperado en prueba")
ax.set_title("Costo esperado: referencia frente a politica optimizada")
plt.tight_layout()
plt.show()

print(f"Politica de costo en prueba: costo={costo_test:,.0f}, % revision={pct_rev_test:.2f}%, "
      f"fraude no detectado={int(fn_test.sum())} de {int(y_test_arr.sum())} "
      f"({fn_test.sum() / y_test_arr.sum() * 100:.2f}%), "
      f"FPR={fp_test.sum() / (y_test_arr == 0).sum() * 100:.3f}%")
print(f"Politica de referencia en prueba: costo={costo_baseline:,.0f} "
      f"(fraude no detectado={int(fn_base.sum())} de {int(y_test_arr.sum())})")

progress["politica_costo"] = {"t_low": t_low, "t_high": t_high, "costo_prueba": float(costo_test),
                               "costo_referencia_prueba": float(costo_baseline)}
save_progress("politica_costo_definida")

Los valores de costo son ilustrativos y estan documentados en la
seccion 11: el multiplicador de fraude no detectado toma la cifra de
LexisNexis citada en el planteamiento del proyecto, y los costos de friccion
y de revision manual son supuestos razonables, no cifras confirmadas por una
institucion real. Lo que si es generalizable es el mecanismo: la politica de
dos umbrales, ajustada sobre validacion con una restriccion explicita de
capacidad de revision, reduce el costo esperado frente a una politica de
umbral unico sin capacidad de revision, porque separa "bloquear" de
"revisar" en vez de forzar una sola decision binaria.

## 18. Monitoreo de degradacion temporal (Objetivo 3)

La particion de prueba se divide en 5 ventanas consecutivas de tamaño
similar. Para cada ventana se recalcula PR-AUC, recall y F1 usando el modelo
estatico de la seccion 15, y se mide el distribution shift comparando la
distribucion del score del modelo en esa ventana contra su distribucion en
validacion, con population stability index (PSI). Se usan las bandas
convencionales de PSI: menor a 0.1 sin cambio relevante, entre 0.1 y 0.25
cambio moderado, mayor a 0.25 cambio importante.

In [ ]:
n_windows = 5
test_dt = df.loc[test_mask, "TransactionDT"]
bordes = np.quantile(test_dt, np.linspace(0, 1, n_windows + 1))
bordes[0] -= 1
window_id = pd.cut(test_dt, bins=bordes, labels=False, include_lowest=True)


def psi(referencia, actual, n_bins=10):
    bins = np.quantile(referencia, np.linspace(0, 1, n_bins + 1))
    bins[0], bins[-1] = -0.001, 1.001
    ref_hist = np.clip(np.histogram(referencia, bins=bins)[0] / len(referencia), 1e-6, None)
    cur_hist = np.clip(np.histogram(actual, bins=bins)[0] / len(actual), 1e-6, None)
    return float(((cur_hist - ref_hist) * np.log(cur_hist / ref_hist)).sum())


filas_ventana = []
for w in range(n_windows):
    idx = window_id[window_id == w].index
    yw = df.loc[idx, "isFraud"].to_numpy()
    pw = best_model.predict_proba(X.loc[idx])[:, 1]
    filas_ventana.append({
        "ventana": w, "n": len(idx),
        "pr_auc": average_precision_score(yw, pw),
        "recall": recall_score(yw, (pw >= t_high).astype(int)),
        "f1": f1_score(yw, (pw >= t_high).astype(int)),
        "psi_score": psi(proba_val, pw),
    })

ventanas_df = pd.DataFrame(filas_ventana)
print(ventanas_df.round(4).to_string(index=False))

progress["ventanas_monitoreo"] = ventanas_df.round(4).to_dict(orient="records")
save_progress("monitoreo_temporal_calculado")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ventanas_df.set_index("ventana")[["pr_auc", "recall", "f1"]].plot(marker="o", ax=axes[0])
axes[0].set_title("PR-AUC, recall y F1 por ventana de prueba")
axes[0].set_xlabel("ventana (0 = mas antigua)")

colors = ["#55A868" if v < 0.1 else "#DD8452" if v < 0.25 else "#C44E52"
          for v in ventanas_df["psi_score"]]
axes[1].bar(ventanas_df["ventana"], ventanas_df["psi_score"], color=colors)
axes[1].axhline(0.1, color="#8C8C8C", linestyle="--", linewidth=1)
axes[1].axhline(0.25, color="#8C8C8C", linestyle="--", linewidth=1)
axes[1].set_title("PSI del score del modelo frente a validacion")
axes[1].set_xlabel("ventana")
plt.tight_layout()
plt.show()

Las ventanas donde el PSI supera 0.1 o 0.25 son las candidatas a
degradacion por cambio en la distribucion de entrada; si en esas mismas
ventanas el PR-AUC tambien cae de forma visible respecto a las primeras
ventanas de prueba, hay evidencia de que el cambio de distribucion esta
acompañado de una perdida real de desempeño predictivo, no solo de un cambio
cosmetico en los datos. Esa combinacion (PSI alto + metrica cayendo) es la
señal que en un sistema en produccion dispararia el reentrenamiento de la
seccion 19.

## 19. Adaptacion: reentrenamiento con ventana deslizante (Objetivo 4)

Para cada una de las 5 ventanas de prueba se entrena un modelo adicional
usando solo los 45 dias mas recientes de datos disponibles antes de esa
ventana, respetando el mismo gap de latencia de etiqueta de la seccion 12.
Se compara su PR-AUC contra el del modelo estatico de la seccion 15, que se
entreno una unica vez con la ventana de entrenamiento original y nunca se
actualiza.

In [ ]:
N_DIAS_VENTANA_RECIENTE = 45
window_starts = df.loc[test_mask].groupby(window_id)["TransactionDT"].min()

filas_adapt = []
for w in range(n_windows):
    idx = window_id[window_id == w].index
    yw = df.loc[idx, "isFraud"].to_numpy()

    inicio_ventana = window_starts.loc[w]
    corte_reciente = inicio_ventana - gap_dias * 86400
    inicio_reciente = corte_reciente - N_DIAS_VENTANA_RECIENTE * 86400
    mask_reciente = (df["TransactionDT"] >= inicio_reciente) & (df["TransactionDT"] < corte_reciente)

    if mask_reciente.sum() < 5000 or df.loc[mask_reciente, "isFraud"].sum() < 20:
        print(f"Ventana {w}: datos recientes insuficientes para reentrenar de forma confiable, se omite.")
        continue

    Xr, yr = X.loc[mask_reciente], y.loc[mask_reciente]
    spw_r = (yr == 0).sum() / max(int(yr.sum()), 1)
    modelo_adapt = lgb.LGBMClassifier(
        device="cpu", objective="binary", n_estimators=500, learning_rate=0.05,
        num_leaves=128, scale_pos_weight=spw_r, random_state=42, n_jobs=-1,
    )
    modelo_adapt.fit(Xr, yr)

    p_static = best_model.predict_proba(X.loc[idx])[:, 1]
    p_adapt = modelo_adapt.predict_proba(X.loc[idx])[:, 1]
    filas_adapt.append({
        "ventana": w, "n_train_reciente": int(mask_reciente.sum()),
        "pr_auc_static": average_precision_score(yw, p_static),
        "pr_auc_adaptive": average_precision_score(yw, p_adapt),
    })

adapt_df = pd.DataFrame(filas_adapt)
print(adapt_df.round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(adapt_df["ventana"], adapt_df["pr_auc_static"], marker="o", label="Estatico", color="#8C8C8C")
ax.plot(adapt_df["ventana"], adapt_df["pr_auc_adaptive"], marker="o", label="Adaptativo (ventana deslizante)",
        color="#55A868")
ax.set_xlabel("ventana de prueba")
ax.set_ylabel("PR-AUC")
ax.set_title("Modelo estatico frente a modelo adaptativo, por ventana")
ax.legend()
plt.tight_layout()
plt.show()

gana_adaptativo = int((adapt_df["pr_auc_adaptive"] > adapt_df["pr_auc_static"]).sum())
print(f"El modelo adaptativo supera al estatico en {gana_adaptativo} de {len(adapt_df)} ventanas evaluadas.")

progress["adaptativo_vs_estatico"] = adapt_df.round(4).to_dict(orient="records")
progress["status"] = "complete"
save_progress("done")

Si el modelo adaptativo iguala o supera al estatico en la mayoria de
las ventanas, sobre todo en aquellas donde la seccion 18 detecto PSI alto,
hay evidencia de que reentrenar con una ventana reciente reduce la
degradacion temporal, y respalda una politica de reentrenamiento disparado
por deteccion de drift en vez de reentrenamiento continuo innecesario, que el
planteamiento del proyecto explicitamente pide evitar por costo. Si el
adaptativo no mejora de forma consistente, la conclusion tambien es valida:
el modelo estatico ya generaliza razonablemente bien dentro del horizonte de
182 dias del dataset, y el reentrenamiento deberia reservarse para cuando el
monitoreo confirme degradacion, no aplicarse por calendario.

## 20. Conclusiones finales

Este pipeline responde a los cuatro objetivos especificos del proyecto:
deteccion de fraude comparando un modelo tradicional contra tres avanzados
(LightGBM, XGBoost, CatBoost) y un benchmark secundario de deep learning (MLP)
bajo split temporal estricto (seccion 15), una politica de decision que
transforma el riesgo en aprobar, revisar o escalar minimizando un costo
esperado bajo restriccion de capacidad (seccion 17), monitoreo de
degradacion temporal por ventanas con PR-AUC, recall, F1 y PSI como medida de
distribution shift (seccion 18), y una comparacion explicita entre
reentrenamiento adaptativo y modelo estatico (seccion 19).

Limitaciones que se mantienen desde el EDA: las variables anonimizadas
impiden una lectura causal de negocio sobre por que ocurre el drift, no solo
que ocurre; y el dataset no ofrece columnas interpretables para una
evaluacion de equidad demografica valida, por lo que esa evaluacion sigue sin
poder realizarse con rigor (seccion 9).

Los supuestos de negocio documentados en la seccion 11 (corte temporal, gap
de latencia, costos de la politica de decision, capacidad de revision) son
el punto donde este sistema es mas facil de ajustar sin rehacer el pipeline:
son parametros, no resultados del analisis, y quedan expuestos como tales
para que se reemplacen apenas se cuente con cifras de negocio confirmadas.